In [6]:
import os
import json5

folder = r"data_sample"

total_requests = 0

for file in os.listdir(folder):
    if file.lower().endswith(".json5"):
        path = os.path.join(folder, file)

        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json5.load(f)

            file_count = 0

            for state in data.values():
                for district in state.get("districts", {}).values():
                    for block in district.get("blocks", {}).values():
                        file_count += len(block.get("panchayats", {}))

            total_requests += file_count
            print(f"{file}: {file_count:,}")

        except Exception as e:
            print(f"{file}: ERROR - {e}")

print("-" * 50)
print(f"TOTAL REQUESTS: {total_requests:,}")

GOA.json5: 191
GUJARAT.json5: 15,108
HARYANA.json5: 6,613
KARNATAKA.json5: 6,103
MADHYA PRADESH.json5: 23,517
MAHA.json5: 29,040
TELANGANA.json5: 13,142
--------------------------------------------------
TOTAL REQUESTS: 93,714


In [8]:
rate = 1.25
6103*rate

7628.75

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from utils import Helper
import urllib3
import warnings
from pathlib import Path
from datetime import datetime
# Suppress the warning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

utils = Helper()

url = "https://report.pmayg.dord.gov.in/netiay/DataAnalytics/PhysicalProgressRpt.aspx"
session = requests.Session()

def get_hidden_fields(soup):
    fields = {}
    for tag in soup.find_all("input", type="hidden"):
        if tag.get("name"):
            fields[tag["name"]] = tag.get("value", "")
    return fields

def parse_panchayat_table(html, state_name, district_name, block_name, panchayat_name, year_code):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", {"id": "ctl00_ContentPlaceHolder1_gvData"})
    if not table:
        # return empty DataFrame with just metadata columns
        return pd.DataFrame(columns=[
            "State","District","Block","Panchayat","Year"
        ])

    df = pd.read_html(str(table))[0]  # keep whatever headers Pandas detects

    # Add metadata
    df["State"] = state_name
    df["District"] = district_name
    df["Block"] = block_name
    df["Panchayat"] = panchayat_name
    df["Year"] = year_code
    return df

def fetch_panchayat_data(
    year_code,
    state_code,
    district_code,
    block_code,
    panchayat_code,
    state_name,
    district_name,
    block_name,
    panchayat_name
):
    def make_soup(response):
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")

    def post(payload):
        response = session.post(
            url,
            data=payload,
            verify=False,
            timeout=60
        )
        response.raise_for_status()
        return response

    # 1. Fresh page
    response = session.get(
        url,
        verify=False,
        timeout=60
    )
    soup = make_soup(response)
    hidden = get_hidden_fields(soup)

    # 2. Select financial year
    payload = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlFinYear",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }

    response = post(payload)
    soup = make_soup(response)
    hidden = get_hidden_fields(soup)

    # 3. Select state
    payload = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlState",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }

    response = post(payload)
    soup = make_soup(response)
    hidden = get_hidden_fields(soup)

    # 4. Select district
    payload = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlDistrict",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }

    response = post(payload)
    soup = make_soup(response)
    hidden = get_hidden_fields(soup)

    # 5. Select block
    payload = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlBlock",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }

    response = post(payload)
    soup = make_soup(response)
    hidden = get_hidden_fields(soup)

    # Confirm that requested Panchayat exists in dropdown
    panchayat_select = soup.find(
        "select",
        id="ctl00_ContentPlaceHolder1_ddlPanchayat"
    )

    if panchayat_select is None:
        raise RuntimeError(
            f"Panchayat dropdown not found for "
            f"{state_name} > {district_name} > {block_name}"
        )

    available_panchayats = {
        option.get("value", ""): option.get_text(strip=True)
        for option in panchayat_select.find_all("option")
    }

    if str(panchayat_code) not in available_panchayats:
        raise ValueError(
            f"Panchayat code {panchayat_code} ({panchayat_name}) "
            f"not available in {block_name}. "
            f"Available codes: {list(available_panchayats.keys())}"
        )

    # 6. Select Panchayat
    payload = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": panchayat_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlPanchayat",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }

    response = post(payload)
    soup = make_soup(response)
    hidden = get_hidden_fields(soup)

    # Verify server retained the requested Panchayat
    selected_option = soup.select_one(
        "#ctl00_ContentPlaceHolder1_ddlPanchayat "
        "option[selected]"
    )

    if selected_option is None:
        selected_option = soup.select_one(
            f'#ctl00_ContentPlaceHolder1_ddlPanchayat '
            f'option[value="{panchayat_code}"]'
        )

    selected_code = (
        selected_option.get("value", "")
        if selected_option
        else ""
    )

    if selected_code != str(panchayat_code):
        raise RuntimeError(
            f"Panchayat selection failed. "
            f"Requested={panchayat_code}, selected={selected_code}"
        )

    # 7. Click Submit
    # IMPORTANT:
    # btnSubmit must be included as a form field.
    # __EVENTTARGET must be blank.
    payload = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": panchayat_code,

        # Actual ASP.NET button click:
        "ctl00$ContentPlaceHolder1$btnSubmit": "Submit",

        "__EVENTTARGET": "",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }

    response = post(payload)

    df = parse_panchayat_table(
        response.text,
        state_name,
        district_name,
        block_name,
        panchayat_name,
        year_code
    )

    if df.empty:
        print(
            f"NO DATA: {state_name} > {district_name} > "
            f"{block_name} > {panchayat_name}"
        )
        return df

    ahead = [
        "Year",
        "State",
        "District",
        "Block",
        "Panchayat"
    ]

    return df[ahead + [
        column for column in df.columns
        if column not in ahead
    ]]
def fetch_block_level_data(year_code, state_code, district_code, block_code,
                           state_name, district_name, block_name):
    # Step 1: GET page
    resp = session.get(url, verify=False)
    hidden = get_hidden_fields(BeautifulSoup(resp.text, "html.parser"))

    # Step 2: Select district
    payload_district = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlDistrict",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp2 = session.post(url, data=payload_district, verify=False)
    hidden2 = get_hidden_fields(BeautifulSoup(resp2.text, "html.parser"))

    # Step 3: Select block
    payload_block = {
        **hidden2,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlBlock",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp3 = session.post(url, data=payload_block, verify=False)
    hidden3 = get_hidden_fields(BeautifulSoup(resp3.text, "html.parser"))

    # Step 4: Trigger panchayat dropdown with "0"
    payload_panchayat = {
        **hidden3,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": "0",
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlPanchayat",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp4 = session.post(url, data=payload_panchayat, verify=False)
    hidden4 = get_hidden_fields(BeautifulSoup(resp4.text, "html.parser"))

    # Step 5: Submit
    payload_submit = {
        **hidden4,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": "0",
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$btnSubmit",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp5 = session.post(url, data=payload_submit, verify=False)

    return parse_block_table(resp5.text, state_name, district_name, block_name, year_code)

def parse_block_table(html, state_name, district_name, block_name, year_code):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", {"id": "ctl00_ContentPlaceHolder1_gvData"})
    if not table:
        return pd.DataFrame(columns=["Year","State","District","Block"])

    # Read the table
    df = pd.read_html(str(table))[0]

    # Add metadata
    df["Year"] = year_code
    df["State"] = state_name
    df["District"] = district_name
    df["Block"] = block_name

    # Reorder columns so metadata comes first
    ahead = ["Year","State","District","Block"]
    df = df[ahead + [c for c in df.columns if c not in ahead]]

    return df


states = ["DAMAN AND DIU.json5"]
for state in states:
    _path_ = Path(state)
    fin_years = {
        # "2016-2017": "2016-2017",
        # "2017-2018": "2017-2018",
        # "2018-2019": "2018-2019",
        # "2019-2020": "2019-2020",
        # "2020-2021": "2020-2021",
        # "2021-2022": "2021-2022",
        # "2022-2023": "2022-2023",
        # "2023-2024": "2023-2024",
        "2024-2025": "2024-2025",
        "2025-2026": "2025-2026",
        # "2026-2027": "2026-2027",
    }


    config = utils.load_json5(state)
    all_dfs = []
    for year_code in fin_years.keys():
        print(year_code)
        for state_code, state_data in config.items():
        
            state_name = state_data["name"]
            print(f"STATE: {state_name}")
            for district_code, district_data in state_data["districts"].items():
                district_name = district_data["name"]
                # if district_name not in select_dists:
                #     continue
                
                print(f">>{district_name}")
                for block_code, block_data in district_data["blocks"].items():
                    
                    # print(f"\t{block_name}")
                    block_name = block_data["name"]
                    panchayat_block = block_data["panchayats"]

                    for panch_code, panch_name in panchayat_block.items():
                        
                        print(f"{panch_code}: {panch_name}")
                        
                        df = fetch_panchayat_data(
                            year_code, state_code, district_code,
                            block_code, panch_code, state_name, district_name, block_name, panch_name
                        )
                        all_dfs.append(df)
                        time.sleep(.4)

    
    _today_ = datetime.now().strftime("%d%m%Y%H%M")
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_excel(f"{_path_.stem}_{_today_}.xlsx", index=False)              